# Lab — Retrieving Data via APIs and Web Scraping (Solution)

**Module 8 — APIs and Web Scraping**  
**Estimated time:** ~60 minutes

This solution notebook demonstrates a complete workflow:
- Collect data from a REST API (with a local fallback)
- Parse JSON into a DataFrame
- Scrape structured data from HTML with BeautifulSoup
- Save raw outputs to `data/raw/`
- Reflect on ethical and legal considerations


## 1) Setup & Environment Check

In [ ]:
import os
from pathlib import Path
import json

import pandas as pd
import requests
from bs4 import BeautifulSoup

print('pandas version:', pd.__version__)
print('requests version:', requests.__version__)

# Create expected folders
Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('notebooks').mkdir(parents=True, exist_ok=True)

print('Folders ready: data/raw, data/processed, notebooks')


## 2) Retrieve Data from an API (Requests)

We'll try a public demo API first:

- `https://jsonplaceholder.typicode.com/posts`

If internet access is unavailable, we fall back to the provided stub file:

- `data/raw/api_posts_stub.json`


In [ ]:
api_url = 'https://jsonplaceholder.typicode.com/posts'
stub_path = Path('data/raw/api_posts_stub.json')

data = None
try:
    response = requests.get(api_url, timeout=30)
    print('Status code:', response.status_code)
    response.raise_for_status()
    data = response.json()
    print('Fetched from live API. Records:', len(data))
except Exception as e:
    print('Live API request failed. Falling back to stub file.')
    print('Reason:', e)
    if not stub_path.exists():
        raise FileNotFoundError('Stub file not found. Expected: data/raw/api_posts_stub.json')
    data = json.loads(stub_path.read_text(encoding='utf-8'))
    print('Loaded from stub. Records:', len(data))


In [ ]:
# Save RAW API response to disk for reproducibility
raw_api_path = Path('data/raw/api_posts.json')
raw_api_path.write_text(json.dumps(data, indent=2), encoding='utf-8')
print('Saved raw API data to:', raw_api_path)


### Optional: Authentication Pattern (Example)

Many APIs require authentication. A safe pattern is to store secrets in environment variables.

**Do not hardcode secrets in notebooks.**


In [ ]:
api_token = os.getenv('API_TOKEN')
headers = {}
if api_token:
    headers = {'Authorization': f'Bearer {api_token}'}
    print('Auth header prepared (token not printed).')
else:
    print('No API_TOKEN found. Skipping authenticated request example.')

# Example usage:
# requests.get('https://api.example.com/secure-endpoint', headers=headers)


## 3) Extract and Structure API Data (Pandas)

In [ ]:
df_posts = pd.DataFrame(data)
df_posts.head()

In [ ]:
# Keep a small, analysis-friendly subset
subset_cols = [c for c in ['userId', 'id', 'title'] if c in df_posts.columns]
df_posts_subset = df_posts[subset_cols].copy()
df_posts_subset.head()

In [ ]:
raw_csv_path = Path('data/raw/api_posts_subset.csv')
df_posts_subset.to_csv(raw_csv_path, index=False)
print('Saved structured API snapshot to:', raw_csv_path)

## 4) Web Scraping with BeautifulSoup (Local HTML)

To keep the lab stable and reproducible, we scrape a **local HTML file**:

- `data/raw/sample_page.html`

This avoids issues caused by changing websites or blocked requests.

In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd().parent
html_path = PROJECT_ROOT / "data" / "raw" / "sample_page.html"

if not html_path.exists():
    raise FileNotFoundError(f"Expected local HTML file not found: {html_path}")

html_text = html_path.read_text(encoding="utf-8")
soup = BeautifulSoup(html_text, "html.parser")


In [ ]:
# Extract product cards
products = []
for item in soup.select('li.product'):
    name = item.select_one('.name').get_text(strip=True)
    price_el = item.select_one('.price')
    price = float(price_el.get_text(strip=True))
    currency = price_el.get('data-currency', 'EUR')
    category_el = item.select_one('.category')
    category = category_el.get_text(strip=True) if category_el else None
    products.append({'name': name, 'price': price, 'currency': currency, 'category': category})

products

In [ ]:
df_products = pd.DataFrame(products)
df_products

In [ ]:
scrape_path = Path('data/raw/scraped_products.csv')
df_products.to_csv(scrape_path, index=False)
print('Saved scraped results to:', scrape_path)

### Bonus: Extract a Table (Shipping)

If the HTML contains a table, you can extract it too (optional).

In [ ]:
shipping_rows = []
for tr in soup.select('#shipping tbody tr'):
    tds = [td.get_text(strip=True) for td in tr.select('td')]
    if len(tds) == 2:
        shipping_rows.append({'region': tds[0], 'days': int(tds[1])})

df_shipping = pd.DataFrame(shipping_rows)
df_shipping

In [ ]:
shipping_path = Path('data/raw/shipping_table.csv')
df_shipping.to_csv(shipping_path, index=False)
print('Saved shipping table to:', shipping_path)

## 5) Ethical and Legal Considerations (Reflection)

**When should an API be preferred over web scraping?**  
- When an API exists, it is typically more stable, structured, and explicitly intended for programmatic access.

**What should you check before scraping a real website?**  
- The site’s Terms of Service, robots.txt, whether the data is public, and whether scraping is permitted.

**How can you reduce impact on a website when scraping?**  
- Use a low request frequency, cache responses, avoid scraping unnecessary pages, and identify yourself responsibly if required.


## 6) Deliverable Check

You should now have created:

- `data/raw/api_posts.json`
- `data/raw/api_posts_subset.csv`
- `data/raw/scraped_products.csv`
- `data/raw/shipping_table.csv` (optional)

Next: commit and push your work to GitHub.

## Version Control

In [ ]:
# Run these commands in your terminal (not inside Python):

# git init
# git add .
# git commit -m "Retrieve data via APIs and web scraping"
# git branch -M main
# git push -u origin main
